Spatial snapping will account for PrevisIA AI roads extracted from satellite imagery having endpoints a few meters away from official OSM roads, made clear when both datasets are visualized in Google Earth Pro and zoomed in.

In [17]:
import geopandas as gpd
import pandas as pd
from shapely.ops import snap, unary_union

In [18]:
# Load the dataset with travel times
file_path = r"C:\Users\308ca\Desktop\Amazonia\Data\combined_roads_paragominas_with_travel_time.geojson"
print(f"Loading dataset from: {file_path}")
combined_gdf = gpd.read_file(file_path)

Loading dataset from: C:\Users\308ca\Desktop\Amazonia\Data\combined_roads_paragominas_with_travel_time.geojson


In [19]:
# Separate official OSM roads from PrevisIA AI traces
osm_backbone = combined_gdf[combined_gdf['source'] == 'OSM (Official Roads)'].copy()
previsia_traces = combined_gdf[combined_gdf['source'] == 'PrevisIA (AI Traces)'].copy()

# Number of fixes to make to align road endpoints
print(f"Official OSM roads: {len(osm_backbone)}")
print(f"PrevisIA traces to snap: {len(previsia_traces)}")

Official OSM roads: 7095
PrevisIA traces to snap: 35791


In [20]:
# Create a spatial index (R-tree - or bounding box of nearby OSM roads) for faster lookups
print("Building spatial index...", flush=True)
osm_sindex = osm_backbone.sindex

Building spatial index...


In [21]:
# Define snapping tolerance
snap_tolerance = 15.0
print(f"Applying fast spatial-indexed snapping (tolerance: {snap_tolerance}m)...", flush=True)

snapped_geometries = []
total_traces = len(previsia_traces)

for idx, geom in enumerate(previsia_traces['geometry']):
    # Buffer the trace slightly by the tolerance to find candidate roads nearby
    search_box = geom.buffer(snap_tolerance)
    
    # Query the index: returns positions of OSM roads that intersect the search box
    possible_matches_idx = osm_sindex.query(search_box, predicate='intersects')
    
    if len(possible_matches_idx) > 0:
        # Extract only the nearby roads using the index matches
        nearby_roads = osm_backbone.iloc[possible_matches_idx]
        local_reference_geom = unary_union(nearby_roads['geometry'].tolist())
        
        # Snap only to these local neighboring roads
        snapped_geom = snap(geom, local_reference_geom, snap_tolerance)
        snapped_geometries.append(snapped_geom)
    else:
        # If no road is within 15 meters, leave the trace geometry as-is
        snapped_geometries.append(geom)
        
    if (idx + 1) % 5000 == 0 or (idx + 1) == total_traces:
        print(f"Processed {idx + 1}/{total_traces} traces...", flush=True)

previsia_traces['geometry'] = snapped_geometries

# Recombine
snapped_combined_gdf = pd.concat([osm_backbone, previsia_traces], ignore_index=True)
snapped_combined_gdf.head(10)

Applying fast spatial-indexed snapping (tolerance: 15.0m)...
Processed 5000/35791 traces...
Processed 10000/35791 traces...
Processed 15000/35791 traces...
Processed 20000/35791 traces...
Processed 25000/35791 traces...
Processed 30000/35791 traces...
Processed 35000/35791 traces...
Processed 35791/35791 traces...


,osm_id,highway,surface,nome,ref,tracktype,source,cat,fonte,road_group,speed_kmh,length_meters,travel_time_hours,geometry
0,31484259.0,trunk,NaN,Belém-Brasília,BR-010,NaN,OSM (Official Roads),NaN,NaN,Paved / Main Roads,15.0,3968.033274,0.264536,"LINESTRING (890707.829 9654954.643, 890870.372..."
1,31484266.0,trunk,asphalt,Rodovia Belém-Brasília,BR-010,NaN,OSM (Official Roads),NaN,NaN,Paved / Main Roads,15.0,516.561873,0.034437,"LINESTRING (889292.355 9584999.365, 889321.847..."
2,31484273.0,trunk,NaN,Belém-Brasília,BR-010,NaN,OSM (Official Roads),NaN,NaN,Paved / Main Roads,15.0,6361.947349,0.424130,"LINESTRING (891613.601 9658817.906, 891633.022..."
3,31484274.0,trunk,NaN,Belém-Brasília,BR-010,NaN,OSM (Official Roads),NaN,NaN,Paved / Main Roads,15.0,20219.877539,1.347992,"LINESTRING (893710.563 9695695.139, 893670.072..."
4,31484287.0,trunk,NaN,Rodovia Bernardo Sayão,BR-010,NaN,OSM (Official Roads),NaN,NaN,Paved / Main Roads,15.0,1712.453310,0.114164,"LINESTRING (892734.807 9635751.038, 892603.575..."
5,31484293.0,trunk,NaN,Rodovia Bernardo Sayão,BR-010,NaN,OSM (Official Roads),NaN,NaN,Paved / Main Roads,15.0,17840.863974,1.189391,"LINESTRING (892402.815 9637431, 892323.013 963..."
6,31484305.0,trunk,asphalt,Avenida Juscelino Kubitschek,BR-010,NaN,OSM (Official Roads),NaN,NaN,Paved / Main Roads,15.0,1083.857943,0.072257,"LINESTRING (889911.252 9717094.542, 889923.436..."
7,31484308.0,trunk,asphalt,Rodovia Bernardo Sayão,BR-010,NaN,OSM (Official Roads),NaN,NaN,Paved / Main Roads,15.0,28151.369438,1.876758,"LINESTRING (890194.688 9608669.9, 890268.92 96..."
8,31484309.0,trunk,NaN,Belém-Brasília,BR-010,NaN,OSM (Official Roads),NaN,NaN,Paved / Main Roads,15.0,10510.670154,0.700711,"LINESTRING (895555 9685347.729, 895469.494 968..."
9,31484327.0,trunk,asphalt,Belém-Brasília,BR-010,NaN,OSM (Official Roads),NaN,NaN,Paved / Main Roads,15.0,4467.385282,0.297826,"LINESTRING (893088.126 9665006.453, 893095.139..."


Processing the snapping/union of about 36,000 entries through a loop takes too long, so switched to indexing to compare a small bounding box of closest roads instead of comparing against the entire city's road network

In [22]:
# Save the clean processed dataset (in UTM meters) for the NetworkX graph step
output_filename = r"C:\Users\308ca\Desktop\Amazonia\Data\combined_roads_paragominas_snapping_union.geojson"
print(f"Saving combined UTM dataset to {output_filename}...")
snapped_combined_gdf.to_file(output_filename, driver="GeoJSON")
print("UTM dataset saved successfully.")

# Save the newly snapped dataset for the graph construction step and mapping later
snapped_path = r"C:\Users\308ca\Desktop\Amazonia\Data\combined_roads_paragominas_snapping_union.geojson"
print(f"Loading snapped dataset from: {snapped_path}")
snapped_gdf = gpd.read_file(snapped_path)

# Reproject to WGS84 Lat/Lon (EPSG:4326) for Google Earth Pro compatibility
print("Reprojecting coordinates to WGS84 (Lat/Lon)...")
snapped_wgs84 = snapped_gdf.to_crs("EPSG:4326")

# Export to a dedicated Google Earth GeoJSON file path
output_wgs84_path = r"C:\Users\308ca\Desktop\Amazonia\Data\combined_roads_paragominas_snapped_wgs84.geojson"
snapped_wgs84.to_file(output_wgs84_path, driver="GeoJSON")

print(f"Exported successfully! Drag and drop this into Google Earth Pro: {output_wgs84_path}")

Saving combined UTM dataset to C:\Users\308ca\Desktop\Amazonia\Data\combined_roads_paragominas_snapping_union.geojson...
UTM dataset saved successfully.
Loading snapped dataset from: C:\Users\308ca\Desktop\Amazonia\Data\combined_roads_paragominas_snapping_union.geojson
Reprojecting coordinates to WGS84 (Lat/Lon)...
Exported successfully! Drag and drop this into Google Earth Pro: C:\Users\308ca\Desktop\Amazonia\Data\combined_roads_paragominas_snapped_wgs84.geojson
